# 01. Model Training & Parameter Estimation

**Goal:**  
Fit a custom Multinomial Naive Bayes classifier on the cleaned Twitter sentiment training dataset. 

**Pipeline Overview:**
1. **Data Ingestion:** Load the preprocessed dataset (`data/processed/train_cleaned.csv`).
2. **Feature Extraction:** Transform raw text into a sparse count matrix using `src/preprocessing.py`.
3. **Model Fitting:** Compute class log-priors and feature log-likelihoods using closed-form NumPy operations in `src/naive_bayes.py`.
4. **Parameter Inspection:** Analyze top features (highest log-likelihoods) per sentiment class to verify model sanity.
5. **Serialization:** Save the fitted vectorizer and trained model state to `models/` for downstream testing.

In [2]:
import sys
import os
import pickle
import pandas as pd
import numpy as np

# Ensure Python can locate modules inside the src/ directory
sys.path.append(os.path.abspath("../src"))

from preprocessing import fit_transform_train
from naive_bayes import MultinomialNaiveBayes

# 1. Load the preprocessed training data
train_data_path = "../data/processed/train_cleaned.csv"
df_train = pd.read_csv(train_data_path)

# Verify data integrity
print(f"Loaded training samples: {len(df_train)}")
print("\nClass Distribution:")
print(df_train['sentiment'].value_counts())

df_train.head()

Loaded training samples: 57297

Class Distribution:
sentiment
Negative    21171
Positive    19078
Neutral     17048
Name: count, dtype: int64


,tweet_content,sentiment
0,im getting on borderlands and i will murder yo...,Positive
1,I am coming to the borders and I will kill you...,Positive
2,im getting on borderlands and i will kill you ...,Positive
3,im coming on borderlands and i will murder you...,Positive
4,im getting on borderlands 2 and i will murder ...,Positive


## 2. Text Vectorization (`CountVectorizer`)

We convert raw tweets into a sparse term-document matrix using `src/preprocessing.py`. 
- **`min_df=5`**: Filters rare tokens/typos appearing in fewer than 5 tweets to reduce noise.
- **`max_df=0.85`**: Ignores overly frequent corpus-wide words appearing in >85% of tweets.
- **Stopwords retained**: Preserves context modifiers and negation signals (e.g., "not", "never").

In [3]:
# Separate features and target labels
X_train_text = df_train['tweet_content']
y_train = df_train['sentiment'].values

# Fit vectorizer on training text and convert to a sparse count matrix
vectorizer, X_train_sparse = fit_transform_train(
    X_train_text, 
    min_df=5, 
    max_df=0.85
)

# Output vocabulary size and feature matrix dimensions
n_samples, n_features = X_train_sparse.shape
print(f"Sparse Matrix Shape: {n_samples} rows (tweets) x {n_features} columns (unique words)")
print(f"Total Non-Zero Elements: {X_train_sparse.nnz}")
print(f"Matrix Sparsity Ratio: {100 * (1 - X_train_sparse.nnz / (n_samples * n_features)):.2f}%")

Sparse Matrix Shape: 57297 rows (tweets) x 14250 columns (unique words)
Total Non-Zero Elements: 940116
Matrix Sparsity Ratio: 99.88%


## 3. Custom Multinomial Naive Bayes Fitting

We fit `MultinomialNaiveBayes` with Laplace smoothing ($\alpha = 1.0$). 
During `fit()`, the model computes:
1. **Log-Priors ($\log P(c)$):** Fraction of training samples per class.
2. **Log-Likelihoods ($\log P(w_i|c)$):** Smoothed word probabilities calculated across sparse column slices.

In [4]:
# Instantiate and fit the model
model = MultinomialNaiveBayes(alpha=1.0)
model.fit(X_train_sparse, y_train)

print("Model training complete.")
print(f"Classes learned: {model.classes_}")

Model training complete.
Classes learned: ['Negative' 'Neutral' 'Positive']


## 4. Inspecting Learned Parameters

To verify what the model learned, we inspect the computed log-priors and map feature log-likelihoods back to their token strings using `vectorizer.get_feature_names_out()`. Higher (less negative) log-likelihood scores indicate words strongly associated with that class.

In [5]:
# Extract parameters
params = model.get_learned_parameters()
feature_names = vectorizer.get_feature_names_out()

# Display Log-Priors
print("--- Log-Priors P(Class) ---")
for c, log_prior in params["log_priors"].items():
    print(f"Class '{c}': {log_prior:.4f} (Prob: {np.exp(log_prior):.4f})")

print("\n--- Top 10 Words by Log-Likelihood per Class ---")
for c in params["classes"]:
    # Retrieve log likelihoods array for class c
    log_likelihoods = params["log_likelihoods"][c]
    
    # Get indices of top 10 highest values
    top_10_idx = np.argsort(log_likelihoods)[-10:][::-1]
    
    print(f"\nClass: [{c.upper()}]")
    for idx in top_10_idx:
        word = feature_names[idx]
        score = log_likelihoods[idx]
        print(f"  {word:<15} | Log-Likelihood: {score:.4f}")

--- Log-Priors P(Class) ---
Class 'Negative': -0.9956 (Prob: 0.3695)
Class 'Neutral': -1.2122 (Prob: 0.2975)
Class 'Positive': -1.0997 (Prob: 0.3330)

--- Top 10 Words by Log-Likelihood per Class ---

Class: [NEGATIVE]
  the             | Log-Likelihood: -3.4150
  to              | Log-Likelihood: -3.8477
  and             | Log-Likelihood: -3.8847
  mention         | Log-Likelihood: -3.9839
  is              | Log-Likelihood: -4.1289
  it              | Log-Likelihood: -4.2336
  of              | Log-Likelihood: -4.3052
  this            | Log-Likelihood: -4.4486
  in              | Log-Likelihood: -4.4629
  you             | Log-Likelihood: -4.4870

Class: [NEUTRAL]
  the             | Log-Likelihood: -3.4921
  to              | Log-Likelihood: -3.8654
  and             | Log-Likelihood: -3.9809
  of              | Log-Likelihood: -4.2600
  in              | Log-Likelihood: -4.3930
  com             | Log-Likelihood: -4.4120
  for             | Log-Likelihood: -4.4374
  it           

## 5. Serializing Artifacts

We persist the fitted `CountVectorizer` and trained `MultinomialNaiveBayes` instance to disk[cite: 1]. The testing notebook will load these files directly to evaluate new validation samples without re-fitting.

In [7]:
# Create models directory if it doesn't exist
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

# 1. Save the trained Naive Bayes model
model_path = os.path.join(models_dir, "naive_bayes_model.pkl")
model.save_model(model_path)
print(f"Saved model state to: {model_path}")

# 2. Save the fitted CountVectorizer
vectorizer_path = os.path.join(models_dir, "vectorizer.pkl")
with open(vectorizer_path, "wb") as f:
    pickle.dump(vectorizer, f)
print(f"Saved fitted vectorizer to: {vectorizer_path}")

Saved model state to: ../models\naive_bayes_model.pkl
Saved fitted vectorizer to: ../models\vectorizer.pkl
